# Configs — 03: Export / Import Catalog Config

**Persona:** Admin

**Goal:** Export all configs from a source (production) catalog, save them locally,
then import them into a target (staging) catalog. Verify that routing resolution
works correctly in the target after import.

**Use case:** Catalog cloning from production to staging — ensures staging mirrors
production driver and routing configuration without manual re-entry.

---

## Prerequisites

- DynaStore running and reachable at `DYNASTORE_BASE_URL`
- Both `SOURCE_CATALOG_ID` and `TARGET_CATALOG_ID` catalogs exist
- `DYNASTORE_ADMIN_TOKEN` is set with admin scope on both catalogs
- Source catalog already has at least one config set (run notebooks 01 or 02 first)

## Environment variables

| Variable | Default | Description |
|---|---|---|
| `DYNASTORE_BASE_URL` | `http://localhost:8080` | API base URL |
| `DYNASTORE_ADMIN_TOKEN` | _(empty)_ | Bearer token for admin operations |
| `SOURCE_CATALOG_ID` | `demo-catalog` | Source (production) catalog |
| `TARGET_CATALOG_ID` | `staging-catalog` | Target (staging) catalog |
| `COLLECTION_ID` | `sentinel2-l2a` | Collection to verify after import |

In [ ]:
import os
import json
import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL          = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
ADMIN_TOKEN       = os.environ.get("DYNASTORE_ADMIN_TOKEN", "")
SOURCE_CATALOG_ID = os.environ.get("SOURCE_CATALOG_ID", "demo-catalog")
COLLECTION_ID     = os.environ.get("COLLECTION_ID", "sentinel2-l2a")

headers = {"Authorization": f"Bearer {ADMIN_TOKEN}"} if ADMIN_TOKEN else {}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60)

print(f"Base URL          : {BASE_URL}")
print(f"Source catalog    : {SOURCE_CATALOG_ID}")
print(f"Collection        : {COLLECTION_ID}")
print(f"Auth header       : {'set' if ADMIN_TOKEN else 'not set'}")


## Edge cases

### Case A — Immutable field already set in target

Some config fields are marked immutable once set (e.g. `partition_type` on a
PostgreSQL driver after the table is created). If the target already has such a
field set, the import should skip it and report it in the `errors` list with a
`skip_immutable` reason — not fail the entire import.

In [ ]:
# Step 1 — "Export" catalog-scoped configs via GET
# (No dedicated /admin/export route; we snapshot the catalog + collection config
# views and assemble our own portable payload.)
resp = client.get(f"/configs/catalogs/{SOURCE_CATALOG_ID}/configs")
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text[:400]}"
catalog_body = resp.json()
catalog_items = catalog_body.get("results", catalog_body if isinstance(catalog_body, list) else [])
catalog_configs = {r["plugin_id"]: r["config"] for r in catalog_items if isinstance(r, dict) and r.get("plugin_id")}

resp = client.get(f"/configs/catalogs/{SOURCE_CATALOG_ID}/collections/{COLLECTION_ID}/configs")
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text[:400]}"
coll_body = resp.json()
coll_items = coll_body.get("results", coll_body if isinstance(coll_body, list) else [])
collection_configs = {r["plugin_id"]: r["config"] for r in coll_items if isinstance(r, dict) and r.get("plugin_id")}

export_payload = {
    "source_catalog_id": SOURCE_CATALOG_ID,
    "catalog_configs": catalog_configs,
    "collection_configs": {COLLECTION_ID: collection_configs},
}
print(f"Catalog configs  : {len(catalog_configs)}")
print(f"Collection configs ({COLLECTION_ID}): {len(collection_configs)}")


In [ ]:
# Step 2 — Save export to a local file
export_path = "export.json"
with open(export_path, "w") as fh:
    json.dump(export_payload, fh, indent=2)

print(f"Export saved to: {export_path}")
with open(export_path) as fh:
    first_64 = fh.read(64)
print(f"First 64 bytes  : {first_64!r}")


In [ ]:
# Step 3 — Re-apply the snapshot via bulk PUT (same-catalog idempotency check)
# Immutable/WriteOnce fields on an existing collection may reject no-op writes;
# accept those as benign.
def apply_bulk(catalog, configs, collection=None):
    path = f"/configs/catalogs/{catalog}/bulk" if not collection else f"/configs/catalogs/{catalog}/collections/{collection}/bulk"
    r = client.put(path, json={"configs": configs})
    assert r.status_code == 200, f"bulk PUT failed {r.status_code}: {r.text[:300]}"
    body = r.json()
    results = body.get("results", [])
    applied = [x for x in results if x.get("status") == "applied"]
    errors  = [x for x in results if x.get("status") == "error"]
    benign  = all(
        ("Immutable" in x.get("detail","")) or ("WriteOnce" in x.get("detail",""))
        for x in errors
    )
    return applied, errors, benign

applied_cat, errs_cat, benign_cat = apply_bulk(SOURCE_CATALOG_ID, catalog_configs)
applied_col, errs_col, benign_col = apply_bulk(SOURCE_CATALOG_ID, collection_configs, COLLECTION_ID)

print(f"Catalog bulk re-apply    : applied={len(applied_cat)} errors={len(errs_cat)} benign={benign_cat}")
print(f"Collection bulk re-apply : applied={len(applied_col)} errors={len(errs_col)} benign={benign_col}")
assert benign_cat and benign_col, "Non-benign errors on idempotent re-apply"


In [ ]:
# Step 4 — Verify routing resolution post-import
resp = client.get(
    f"/configs/catalogs/{SOURCE_CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionRoutingConfig/effective"
)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text[:400]}"
effective = resp.json()
resolved  = effective.get("resolved", {})
sources   = effective.get("sources", {})
print("Effective routing:")
print(json.dumps(resolved, indent=2))
print(f"Source levels: {set(sources.values())}")
client.close()
